Logs

In [ ]:
%run Utils_Log

In [ ]:
setup_log("Ingesta_mares")

In [3]:
#%pip install "numpy<2.0.0" "copernicusmarine>=2.0" --quiet

StatementMeta(, 1237a5fb-d124-4430-a11f-320a5b338233, 11, Finished, Available, Finished, False)

Found existing installation: numpy 1.26.4
Not uninstalling numpy at /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages, outside environment /nfs4/pyenv-84376b2b-c6c6-47de-a11c-e7c0be28966c
Can't uninstall 'numpy'. No files were found to uninstall.
Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fsspec-wrapper 0.1.15 requires PyJWT>=2.6.0, but you have pyjwt 2.4.0 which is incompatible.
mlflow-skinny 2.12.2 requires packaging<25, but you have packaging 26.2 which is incompatible.
nni 3.0 requires filelock<3.12, but you have filelock 3.13.1 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



# Descarga de Datos Oceanográficos – Bancos de Pesca
## A Coruña & Pontevedra · Copernicus Marine Service

Este notebook descarga datos de **salinidad**, **temperatura** y **velocidad de corrientes** para todos los bancos de pesca de las provincias de A Coruña y Pontevedra, utilizando el servicio Copernicus Marine (CMEMS).

**Variables descargadas:**
- Salinidad (`so`) → `cmems_mod_glo_phy-so_anfc_0.083deg_P1D-m`
- Temperatura (`thetao`) → `cmems_mod_glo_phy-thetao_anfc_0.083deg_P1D-m`
- Corrientes U/V (`uo`, `vo`) → `cmems_mod_glo_phy-cur_anfc_0.083deg_P1D-m`

Logs

In [6]:
import copernicusmarine
from notebookutils import mssparkutils
from pathlib import Path

# AUTENTICACIÓN SEGURA
CMEMS_USER = "sebastianramostueros.srt@gmail.com" 
CMEMS_PASSWORD = "Sebas.300697"

copernicusmarine.login(username=CMEMS_USER, password=CMEMS_PASSWORD, force_overwrite=True)
log(f"Conectando con Azure Key Vault para recuperar la contraseña del usuario: {CMEMS_USER}")

# DEFINICIÓN FÍSICA DE LOS BANCOS
BANCOS_PESCA = [
    {"nombre": "Zona I - Vigo",          "latitud": 42.22356, "longitud": -8.82484},
    {"nombre": "Zona II - Pontevedra",   "latitud": 42.36464, "longitud": -8.81874},
    {"nombre": "Zona III - Arousa",      "latitud": 42.50921, "longitud": -8.94458},
    {"nombre": "Zona IV - Muros",        "latitud": 42.69498, "longitud": -9.08367},
    {"nombre": "Zona V - Fisterra",      "latitud": 42.85957, "longitud": -9.21355},
    {"nombre": "Zona VI - Costa da Morte", "latitud": 43.229,   "longitud": -9.05312},
    {"nombre": "Zona VII - Coruña-Ferrol", "latitud": 43.4249,  "longitud": -8.3557},
    {"nombre": "Zona VIII - Cedeira",    "latitud": 43.76145, "longitud": -8.00771},
    {"nombre": "Zona IX - Mariña",       "latitud": 43.71027, "longitud": -7.24132},
]

# Variables de la API
DATASET_ID = "cmems_mod_glo_phy_my_0.083deg_P1D-m"
VARIABLES = ["so", "thetao", "uo", "vo"]
START = "2015-01-01T00:00:00"
END = "2026-04-28T00:00:00"
DEPTH_MIN = 0.49402499198913574
DEPTH_MAX = 0.49402499198913574

OUTPUT_DIR = Path("/lakehouse/default/Files/Bronze/Oceanografia") ##### creacion de los .nc dentro de mi bronze
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 3. EXTRACCIÓN (Bucle de descarga)
for banco in BANCOS_PESCA:
    nombre = banco["nombre"]
    banco_dir = OUTPUT_DIR / nombre
    banco_dir.mkdir(parents=True, exist_ok=True)
    ruta_destino = f"Files/Bronze/Oceanografia/{nombre}/{nombre}.nc"

    try;
        mssparkutils.fs.head(ruta_destino, 1)
        log(f"Omitiendo archivos {nombre}: el .nc ya existe")
    except Exception:
        log(f"Descargando archivos {nombre} en la carpeta")
        copernicusmarine.subset(
            dataset_id=DATASET_ID, variables=VARIABLES,
            minimum_longitude=banco["longitud"], maximum_longitude=banco["longitud"],
            minimum_latitude=banco["latitud"], maximum_latitude=banco["latitud"],
            start_datetime=START, end_datetime=END,
            minimum_depth=DEPTH_MIN, maximum_depth=DEPTH_MAX,
            output_filename=f"{nombre}.nc", output_directory=str(banco_dir), overwrite=True
        )

log("Creado el CSV de Ingesta_mares")


StatementMeta(, 1237a5fb-d124-4430-a11f-320a5b338233, 20, Finished, Available, Finished, False)

INFO - 2026-06-02T12:06:05Z - Credentials file stored in /home/trusted-service-user/.copernicusmarine/.copernicusmarine-credentials.
INFO - 2026-06-02T12:06:06Z - Selected dataset version: "202311"
INFO - 2026-06-02T12:06:06Z - Selected dataset part: "default"


  0%|          | 0/24 [00:00<?, ?it/s]

INFO - 2026-06-02T12:06:18Z - Selected dataset version: "202311"
INFO - 2026-06-02T12:06:18Z - Selected dataset part: "default"


  0%|          | 0/24 [00:00<?, ?it/s]

...

RejectSilentExecuteRequest: Livy session has failed. Error code: RejectSilentExecuteRequest. Rejected silent execute_request as there is no active session.

StatementMeta(, , -1, Finished, , Finished, True)

RejectSilentExecuteRequest: Livy session has failed. Error code: RejectSilentExecuteRequest. Rejected silent execute_request as there is no active session.

StatementMeta(, , -1, Finished, , Finished, True)

RejectSilentExecuteRequest: Livy session has failed. Error code: RejectSilentExecuteRequest. Rejected silent execute_request as there is no active session.

In [ ]:
#mssparkutils.session.stop()